# Layer 3 — Phase 1 v3 (Setup + AIOps Challenge 2020)

## v2 → v3 변경사항
1. zip 못 찾으면 **즉시 FileNotFoundError 발생** (조용히 None 두지 않음)
2. `FAULT_TIME_COL`을 **None으로 시작** + 명시 안 하면 ValueError
3. normal day 선택 정책 **markdown 명시** (Phase 2에서 재선정)

## 작업 목표
- AIOps Challenge 2020 zip 압축 해제
- fault 라벨 분석 → 장애 날짜 우선 추출
- 3종 메트릭 구조 파악
- 6개 핵심 KPI 매핑 후보 자동 식별
- Phase 2로 넘길 메타정보 저장

## ⚠️ 사전 준비
1. https://drive.google.com/file/d/1nkEsD1g7THm_T58KwUQZ7o-b174fdx-n/view 에서 zip 다운로드
2. 본인 Google Drive에 폴더 생성: `MyDrive/layer3_data/raw/`
3. 받은 zip을 위 폴더에 업로드
4. 이 노트북을 Colab에서 열기

## 0. Colab 환경 셋업

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/layer3_data'
RAW = f'{ROOT}/raw'
WORK = f'{ROOT}/work'
OUT = f'{ROOT}/processed'

for p in [ROOT, RAW, WORK, OUT]:
    os.makedirs(p, exist_ok=True)

os.chdir(WORK)
print(f'작업 디렉토리: {os.getcwd()}')
print(f'RAW: {RAW}')
print(f'WORK: {WORK}')
print(f'OUT: {OUT}')

In [ ]:
# 라이브러리
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import zipfile
import json
import glob
from collections import Counter
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)

# 한글 폰트
!apt -qq install fonts-nanum > /dev/null 2>&1
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

print('라이브러리 로드 완료')

## 1. 업로드된 zip 확인 (v3: 즉시 중단)

In [ ]:
raw_files = sorted(os.listdir(RAW))
print(f'RAW 폴더 안 파일들:')
for f in raw_files:
    fp = os.path.join(RAW, f)
    sz = os.path.getsize(fp) / 1e6
    print(f'  {f} ({sz:.1f} MB)')

candidates = [
    f for f in raw_files
    if f.endswith('.zip') and ('aiops' in f.lower() or '挑战' in f or '2020' in f)
]

# v3: 못 찾으면 즉시 중단
if not candidates:
    raise FileNotFoundError(
        f"AIOps Challenge 2020 zip을 {RAW}에서 찾지 못했습니다. "
        "파일명에 'aiops', '2020', 또는 '挑战'이 포함된 zip을 업로드했는지 확인하세요."
    )

AIOPS_ZIP = os.path.join(RAW, candidates[0])
print(f'\n사용할 zip: {AIOPS_ZIP}')

## 2. 외부 zip 압축 해제

In [ ]:
EXTRACT_DIR = os.path.join(WORK, 'aiops_extracted')
os.makedirs(EXTRACT_DIR, exist_ok=True)

if not os.listdir(EXTRACT_DIR):
    print('외부 zip 압축 해제 중...')
    with zipfile.ZipFile(AIOPS_ZIP, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('완료')
else:
    print(f'이미 풀려있음: {EXTRACT_DIR}')

# 구조 확인
for root, dirs, files in os.walk(EXTRACT_DIR):
    level = root.replace(EXTRACT_DIR, '').count(os.sep)
    if level > 2: continue
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        for f in sorted(files)[:5]:
            print(f'{indent}  {f}')
        if len(files) > 5:
            print(f'{indent}  ... and {len(files)-5} more')

## 3. fault 라벨 먼저 분석

In [ ]:
label_files = glob.glob(f'{EXTRACT_DIR}/**/故障*.csv', recursive=True)
if not label_files:
    all_csvs = glob.glob(f'{EXTRACT_DIR}/**/*.csv', recursive=True)
    label_files = [f for f in all_csvs if 'metric' not in f.lower() and 'kpi' not in f.lower()]

print(f'발견된 라벨 파일 후보:')
for f in label_files:
    print(f'  {f}')

if not label_files:
    raise FileNotFoundError('fault 라벨 파일 못 찾음. 압축 해제 결과 확인 필요.')

In [ ]:
faults = None
for enc in ['utf-8', 'gbk', 'utf-8-sig', 'gb18030']:
    try:
        faults = pd.read_csv(label_files[0], encoding=enc)
        print(f'성공 (인코딩: {enc})')
        break
    except UnicodeDecodeError:
        continue

if faults is None:
    raise UnicodeDecodeError('utf-8/gbk/utf-8-sig/gb18030 모두 실패. 다른 인코딩 시도 필요.')

print(f'\n라벨 데이터: {len(faults)} 행')
print(f'컬럼: {faults.columns.tolist()}')
display(faults.head(10))

In [ ]:
# 컬럼별 통계
print('--- 컬럼별 통계 ---')
for col in faults.columns:
    n_unique = faults[col].nunique()
    print(f'\n[{col}] unique 값: {n_unique}, dtype: {faults[col].dtype}')
    if n_unique <= 20:
        print(faults[col].value_counts())
    else:
        print(f'sample: {faults[col].dropna().head(3).tolist()}')

### 3-1. fault time column 후보 출력

In [ ]:
time_keywords = ['time', 'date', 'start', 'end', '时间', '日期', '开始', '结束', 'timestamp']
time_like_cols = [
    c for c in faults.columns
    if any(k in str(c).lower() for k in time_keywords)
]
print(f'시간 관련 컬럼 후보: {time_like_cols}')
print('\n각 후보의 샘플 값:')
for c in time_like_cols:
    sample_vals = faults[c].dropna().head(3).tolist()
    print(f'  [{c}]: {sample_vals}')

print('\n👉 위 출력을 보고, 다음 셀에서 FAULT_TIME_COL을 fault 시작 시각 컬럼으로 직접 설정하세요.')

In [ ]:
# v3: 명시적 None으로 시작. 사용자가 반드시 직접 설정해야 함.
FAULT_TIME_COL = None

# 예시 (위 출력 보고 정확한 컬럼명으로 수정):
# FAULT_TIME_COL = '故障开始时间'
# FAULT_TIME_COL = 'start_time'
# FAULT_TIME_COL = '时间'

if FAULT_TIME_COL is None:
    raise ValueError(
        'FAULT_TIME_COL이 설정되지 않았습니다. 위 셀의 시간 컬럼 후보를 보고, '
        '이 셀에서 fault 시작 시각 컬럼명을 직접 지정하세요. '
        '주의: 첫 번째 후보가 종료 시각이거나 보고 시각일 수 있음.'
    )

if FAULT_TIME_COL not in faults.columns:
    raise ValueError(
        f'FAULT_TIME_COL="{FAULT_TIME_COL}"이 fault 컬럼에 없습니다. '
        f'사용 가능: {faults.columns.tolist()}'
    )

print(f'사용할 fault 시간 컬럼: {FAULT_TIME_COL}')

### 3-2. fault 발생 날짜 추출

In [ ]:
faults[FAULT_TIME_COL] = pd.to_datetime(faults[FAULT_TIME_COL], errors='coerce')
n_parsed = faults[FAULT_TIME_COL].notna().sum()
n_total = len(faults)
print(f'시간 파싱 성공: {n_parsed}/{n_total}')

if n_parsed == 0:
    raise ValueError(
        f'{FAULT_TIME_COL} 컬럼에서 시간 파싱이 모두 실패. 컬럼이 시간 형식이 맞는지 확인.'
    )

# zip 파일명 형식과 매칭 (2020_04_11)
fault_days = sorted(faults[FAULT_TIME_COL].dropna().dt.strftime('%Y_%m_%d').unique())
print(f'\nfault 발생 일자: 총 {len(fault_days)}일')
for d in fault_days[:20]:
    n_faults_on_day = (faults[FAULT_TIME_COL].dt.strftime('%Y_%m_%d') == d).sum()
    print(f'  {d}: {n_faults_on_day} 건')
if len(fault_days) > 20:
    print(f'  ... 외 {len(fault_days)-20}일')

## 4. fault 날짜 + 정상 비교용 날짜로 zip 풀기

**v3 정책 명시 (Phase 2에서 재선정):**
> 현재 normal day는 EDA 목적의 임시 선택이며, Phase 2에서는 fault label 기준으로 incident interval과 겹치지 않는 구간을 normal window로 재선정한다. 즉 fault day ±1일 등은 normal에서 제외하는 정밀한 선택이 Phase 2에서 들어감.

In [ ]:
inner_zips = sorted(glob.glob(f'{EXTRACT_DIR}/**/*.zip', recursive=True))
all_days_in_zip = [os.path.splitext(os.path.basename(z))[0] for z in inner_zips]
print(f'전체 일별 zip: {len(inner_zips)}개')
print(f'날짜 범위: {all_days_in_zip[0] if all_days_in_zip else "없음"} ~ {all_days_in_zip[-1] if all_days_in_zip else "없음"}')

In [ ]:
N_FAULT_DAYS = 3
N_NORMAL_DAYS = 2

fault_zips = [z for z in inner_zips 
              if any(d in os.path.basename(z) for d in fault_days)]
print(f'fault 있는 일별 zip: {len(fault_zips)}개')

fault_zips_to_extract = fault_zips[:N_FAULT_DAYS]
normal_zips = [z for z in inner_zips if z not in fault_zips]
normal_zips_to_extract = normal_zips[:N_NORMAL_DAYS]

zips_to_extract = fault_zips_to_extract + normal_zips_to_extract

print(f'\n추출 대상 zip ({len(zips_to_extract)}개):')
for z in fault_zips_to_extract:
    print(f'  [fault] {os.path.basename(z)}')
for z in normal_zips_to_extract:
    print(f'  [normal] {os.path.basename(z)}')

if not zips_to_extract:
    raise ValueError(
        '추출할 zip 없음. fault_days와 inner_zips 매칭 실패. '
        f'fault_days: {fault_days[:3]}..., inner_zips: {[os.path.basename(z) for z in inner_zips[:3]]}...'
    )

In [ ]:
DAILY_DIR = os.path.join(WORK, 'aiops_daily')
os.makedirs(DAILY_DIR, exist_ok=True)

for zp in zips_to_extract:
    day = os.path.splitext(os.path.basename(zp))[0]
    target = os.path.join(DAILY_DIR, day)
    if os.path.exists(target) and os.listdir(target):
        print(f'스킵 (이미 있음): {day}')
        continue
    os.makedirs(target, exist_ok=True)
    print(f'{day} 풀기...')
    with zipfile.ZipFile(zp, 'r') as z:
        z.extractall(target)

print('\n압축 해제 완료')
print(f'풀린 일자: {sorted(os.listdir(DAILY_DIR))}')

## 5. 3종 메트릭 폴더 식별

In [ ]:
first_day = sorted(os.listdir(DAILY_DIR))[0]
first_day_path = os.path.join(DAILY_DIR, first_day)
print(f'분석할 날짜: {first_day}')

for sub in os.listdir(first_day_path):
    sub_path = os.path.join(first_day_path, sub)
    if os.path.isdir(sub_path):
        files = os.listdir(sub_path)
        total_sz = sum(os.path.getsize(os.path.join(sub_path, f)) for f in files) / 1e6
        print(f'  {sub}/  ({len(files)} 파일, 합계 {total_sz:.1f} MB)')

In [ ]:
biz_dir = None
infra_dir = None
trace_dir = None

for sub in os.listdir(first_day_path):
    sub_path = os.path.join(first_day_path, sub)
    if not os.path.isdir(sub_path):
        continue
    if '业务' in sub or 'business' in sub.lower():
        biz_dir = sub_path
    elif '平台' in sub or 'platform' in sub.lower() or 'infra' in sub.lower():
        infra_dir = sub_path
    elif '调用链' in sub or 'trace' in sub.lower():
        trace_dir = sub_path

print(f'business dir: {biz_dir}')
print(f'infra dir: {infra_dir}')
print(f'trace dir: {trace_dir}')

if not all([biz_dir, infra_dir, trace_dir]):
    print('\n⚠️ 3종 폴더 모두 식별 실패. 폴더명 확인 후 위 변수 수동 설정 필요.')
    print(f'사용 가능 폴더: {[s for s in os.listdir(first_day_path) if os.path.isdir(os.path.join(first_day_path, s))]}')

## 6. 각 메트릭 샘플

In [ ]:
biz_sample = None
if biz_dir:
    biz_files = sorted(os.listdir(biz_dir))
    print(f'business 파일 수: {len(biz_files)}')
    print(f'예시: {biz_files[:5]}')
    biz_sample = pd.read_csv(os.path.join(biz_dir, biz_files[0]), nrows=10000)
    print(f'\n샘플: {biz_files[0]}, {len(biz_sample)} 행, 컬럼 {len(biz_sample.columns)}개')
    display(biz_sample.head(10))

In [ ]:
infra_sample = None
if infra_dir:
    infra_files = sorted(os.listdir(infra_dir))
    print(f'infrastructure 파일 수: {len(infra_files)}')
    print(f'예시: {infra_files[:5]}')
    infra_sample = pd.read_csv(os.path.join(infra_dir, infra_files[0]), nrows=10000)
    print(f'\n샘플: {infra_files[0]}, {len(infra_sample)} 행, 컬럼 {len(infra_sample.columns)}개')
    display(infra_sample.head(10))

In [ ]:
trace_sample = None
if trace_dir:
    trace_files = sorted(os.listdir(trace_dir))
    print(f'trace 파일 수: {len(trace_files)}')
    print(f'예시: {trace_files[:5]}')
    trace_sample = pd.read_csv(os.path.join(trace_dir, trace_files[0]), nrows=10000)
    print(f'\n샘플 (첫 10000행만): {trace_files[0]}, 컬럼 {len(trace_sample.columns)}개')
    display(trace_sample.head(10))

## 7. 컬럼 후보 자동 탐색

In [ ]:
def find_candidate_columns(df, keywords):
    result = {}
    for key in keywords:
        matches = [c for c in df.columns if key.lower() in str(c).lower()]
        if matches:
            result[key] = matches
    return result

keywords = [
    'time', 'timestamp', 'start', 'end', 'duration', 'latency',
    'success', 'error', 'fail', 'status', 'count', 'rate',
    'cpu', 'memory', 'mem', 'network', 'net', 'value', 'kpi',
    '时间', '时长', '成功', '失败', '状态', '次数', '率', '指标', '名'
]

column_candidates = {}

if biz_sample is not None:
    column_candidates['business'] = {
        'columns': biz_sample.columns.tolist(),
        'keyword_matches': find_candidate_columns(biz_sample, keywords)
    }
if infra_sample is not None:
    column_candidates['infrastructure'] = {
        'columns': infra_sample.columns.tolist(),
        'keyword_matches': find_candidate_columns(infra_sample, keywords)
    }
if trace_sample is not None:
    column_candidates['trace'] = {
        'columns': trace_sample.columns.tolist(),
        'keyword_matches': find_candidate_columns(trace_sample, keywords)
    }
if faults is not None:
    column_candidates['fault_label'] = {
        'columns': faults.columns.tolist(),
        'keyword_matches': find_candidate_columns(faults, keywords)
    }

for src, info in column_candidates.items():
    print(f'\n=== {src.upper()} ===')
    print(f'전체 컬럼 ({len(info["columns"])}개): {info["columns"]}')
    print(f'\nkeyword 매칭:')
    if info['keyword_matches']:
        for kw, cols in info['keyword_matches'].items():
            print(f'  [{kw}] → {cols}')
    else:
        print('  (매칭 없음)')

## 8. 6개 핵심 KPI 매핑 표

In [ ]:
kpi_mapping = pd.DataFrame([
    {'KPI': 'P99 latency', 'Source': 'trace',
     'Method': 'trace duration의 1분 단위 P99', 'Status': 'TBD'},
    {'KPI': 'timeout count', 'Source': 'trace',
     'Method': 'duration > timeout threshold 비율', 'Status': 'TBD'},
    {'KPI': 'API error rate', 'Source': 'business',
     'Method': '1 - success_rate 또는 status >= 500', 'Status': 'TBD'},
    {'KPI': 'data ingestion delay', 'Source': 'derived',
     'Method': '예상 timestamp 간격 대비 실제 지연', 'Status': 'derived'},
    {'KPI': 'monitoring log missing rate', 'Source': 'derived',
     'Method': '예상 로그 수 대비 실제 비율', 'Status': 'derived'},
    {'KPI': 'cloud incident flag', 'Source': '故障整理',
     'Method': 'fault interval을 1분 단위 0/1로', 'Status': '직접 매핑'},
])
display(kpi_mapping)

## 9. Phase 2로 넘길 메타정보 저장

In [ ]:
faults.to_parquet(os.path.join(OUT, 'aiops_fault_labels.parquet'), index=False)
print(f'저장: aiops_fault_labels.parquet ({len(faults)} 행)')

kpi_mapping.to_csv(os.path.join(OUT, 'kpi_mapping_aiops.csv'), index=False)
print(f'저장: kpi_mapping_aiops.csv')

data_index = {
    'aiops_extract_dir': EXTRACT_DIR,
    'aiops_daily_dir': DAILY_DIR,
    'aiops_days_extracted': sorted(os.listdir(DAILY_DIR)),
    'business_subdir_name': os.path.basename(biz_dir) if biz_dir else None,
    'infra_subdir_name': os.path.basename(infra_dir) if infra_dir else None,
    'trace_subdir_name': os.path.basename(trace_dir) if trace_dir else None,
    'business_dir_full_path': biz_dir,
    'infra_dir_full_path': infra_dir,
    'trace_dir_full_path': trace_dir,
    'label_file': label_files[0] if label_files else None,
    'fault_time_col': FAULT_TIME_COL,
    'fault_days': fault_days,
    'fault_zips_extracted': [os.path.basename(z) for z in fault_zips_to_extract],
    'normal_zips_extracted': [os.path.basename(z) for z in normal_zips_to_extract],
}
with open(os.path.join(OUT, 'aiops_data_index.json'), 'w', encoding='utf-8') as f:
    json.dump(data_index, f, ensure_ascii=False, indent=2, default=str)
print('저장: aiops_data_index.json')

with open(os.path.join(OUT, 'aiops_column_candidates.json'), 'w', encoding='utf-8') as f:
    json.dump(column_candidates, f, ensure_ascii=False, indent=2, default=str)
print('저장: aiops_column_candidates.json')

## ✅ Part 1 v3 완료 → Part 2로

OUT 폴더에 4개 파일 저장 확인 후 Part 2 노트북 진행.